In [ ]:
# Databricks notebook source
# 02_clean_dgz

import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(taskKey="00_check_source_changes", key="should_run", default="true")
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    output_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date",
        "Date_month",
        "Date_week",
        "Pathogen",
        "Result",
    ]
    pdf = pdf.reindex(columns=output_cols).copy()

    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")

if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping DGZ")

raw = pd.read_excel(source_path("MTA_DECIDE_Ugent2025.xlsx"), engine="openpyxl")
aero_raw = pd.read_excel(source_path("DECIDE_MTA_UGENT_BAC_AERO_14nov2022.xlsx"), engine="openpyxl")
myco_raw = pd.read_excel(source_path("DECIDE_MTA_UGENTBAC_MYCO_14nov2022.xlsx"), engine="openpyxl")

aero = aero_raw.rename(columns={"Dossiernummer": "Filenumber", "KIEMSTAAL IDENTIFICATIE": "Pathogen_identification", "KIEMSTAAL RESULTAAT": "Pathogen_result", "Staalnummer": "Samplenumber"})
aero = aero.assign(Parameter_code="BAC_AERO", Result="OK")[["Filenumber", "Pathogen_identification", "Pathogen_result", "Parameter_code", "Samplenumber", "Result"]]
aero = aero[aero["Pathogen_identification"].isin(["Pasteurella multocida", "Mannheimia haemolytica", "Histophilus somni", "Mycoplasma bovis"])].drop_duplicates()
for col in ["Filenumber", "Samplenumber"]:
    aero[col] = aero[col].apply(sha256_hash)

myco = myco_raw.rename(columns={"Dossiernummer": "Filenumber", "KIEMSTAAL IDENTIFICATIE": "Pathogen_identification", "KIEMSTAAL RESULTAAT": "Mycoplasma_result", "Staalnummer": "Samplenumber"})
myco = myco.assign(Parameter_code="BAC_MYCOPLASMA", Result="OK")[["Filenumber", "Pathogen_identification", "Mycoplasma_result", "Parameter_code", "Samplenumber", "Result"]]
myco = myco[myco["Pathogen_identification"].isin(["Mycoplasma bovis"])].drop_duplicates()
for col in ["Filenumber", "Samplenumber"]:
    myco[col] = myco[col].apply(sha256_hash)

df = raw.rename(columns={"Dossiernummer": "Filenumber", "Staalnummer": "Samplenumber", "Staaltype": "Sample_type", "PARAMETER_CODE": "Parameter_code", "Onderzoek": "Pathogen", "Resultaat": "Result", "Creatiedatum": "Date", "Postcode": "Postal_code", "ANON_ID": "Farm_ID"})
df["Country"] = "Belgium"
df["Diagnostic_test"] = np.where(df["Parameter_code"].isin(["BAC_AERO", "BAC_MYCOPLASMA"]), "Culture", "PCR")
df["Lab_reference"] = "1"
df["Sample_type"] = np.select(
    [df["Sample_type"].eq("RU Broncho-alveolar lavage (BAL)"), df["Sample_type"].eq("RU Anderen"), df["Sample_type"].isin(["RU Swabs", "RU Swab", "RU Neusswab", "RU Neusswabs"]), df["Sample_type"].isin(["RU Kadaver", "RU Organen"])],
    ["BAL", "Unknown", "Swab", "Autopsy"],
    default="Missing",
)
df["Breed"] = np.select(
    [df["Bedrijfstype"].eq("VCALF"), df["MEAT"].isna(), pd.to_numeric(df["MEAT"], errors="coerce") / pd.to_numeric(df["TOTAL"], errors="coerce") > 0.9, pd.to_numeric(df["MILK"], errors="coerce") / pd.to_numeric(df["TOTAL"], errors="coerce") > 0.9],
    ["Veal", "Unknown", "Beef", "Dairy"],
    default="Mixed",
)
pathogen_map = {
    "AD Pasteurella multocida Ag (PCR)": "Pasteurella multocida", "AD Pasteurella multocida Ag pool (PCR)": "Pasteurella multocida", "AD P. multocida Ag (PCR)": "Pasteurella multocida", "AD P. multocida Ag pool (PCR)": "Pasteurella multocida",
    "AD Mannheimia haemolytica Ag (PCR)": "Mannheimia haemolytica", "AD Mannheimia haemolytica Ag pool (PCR)": "Mannheimia haemolytica",
    "RU PI3 Ag (PCR)": "PI3", "RU PI3 Ag pool (PCR)": "PI3", "RU BRSV Ag (PCR)": "BRSV", "RU BRSV Ag pool (PCR)": "BRSV",
    "AD Histophilus somnus (PCR)": "Histophilus somni", "AD Histophilus somnus Ag (PCR)": "Histophilus somni", "AD Histophilus somnus Ag pool (PCR)": "Histophilus somni", "AD Histophilus somni Ag (PCR)": "Histophilus somni", "AD Histophilus somni Ag pool (PCR)": "Histophilus somni",
    "RU Mycoplasma bovis (PCR)": "Mycoplasma bovis", "RU Mycoplasma bovis Ag pool (PCR)": "Mycoplasma bovis", "RU Mycoplasma bovis Ag (PCR)": "Mycoplasma bovis",
    "AD Corona Ag (PCR)": "BCV", "AD Corona Ag pool (PCR)": "BCV",
}
df["Pathogen"] = df["Pathogen"].replace(pathogen_map)
pc = pd.to_numeric(df["Postal_code"], errors="coerce")
df["Province"] = np.select(
    [pc.between(1000, 1299), pc.between(1300, 1499), pc.between(1500, 1999), pc.between(3000, 3499), pc.between(2000, 2999) | pc.between(3500, 3999), pc.between(4000, 4999), pc.between(5000, 5999), pc.between(6000, 6599) | pc.between(7000, 7999), pc.between(6600, 6999), pc.between(8000, 8999)],
    ["Brussels", "Walloon Brabant", "Flemish Brabant", "Antwerp", "Limburg", "Li?ge", "Namur", "Hainaut", "Luxembourg", "West Flanders"],
    default="East Flanders",
)
keep = ["Filenumber", "Diagnostic_test", "Samplenumber", "Country", "Lab_reference", "Sample_type", "Breed", "Parameter_code", "Result", "Pathogen", "Date", "Province", "Farm_ID"]
df = df[keep].drop_duplicates()
for col in ["Filenumber", "Samplenumber", "Farm_ID"]:
    df[col] = df[col].apply(sha256_hash)

samples = pd.DataFrame({"Result": ["OK", "OK", "OK", "OK"], "Parameter_code": ["BAC_AERO", "BAC_AERO", "BAC_AERO", "BAC_MYCOPLASMA"], "Diagnostic_test": ["Culture", "Culture", "Culture", "Culture"], "Pathogen_identification": ["Pasteurella multocida", "Mannheimia haemolytica", "Histophilus somni", "Mycoplasma bovis"]})
barometer = df.merge(samples, on=["Diagnostic_test", "Result", "Parameter_code"], how="left")
barometer = barometer.merge(aero, on=["Filenumber", "Samplenumber", "Result", "Parameter_code", "Pathogen_identification"], how="left")
barometer = barometer.merge(myco, on=["Filenumber", "Samplenumber", "Result", "Parameter_code", "Pathogen_identification"], how="left")
barometer["Date"] = pd.to_datetime(barometer["Date"], errors="coerce")
barometer["Date_month"] = month_start(barometer["Date"])
barometer["Date_week"] = week_start(barometer["Date"], week_start="sunday")
short = {"Pasteurella multocida": "PM", "Histophilus somni": "HS", "Mannheimia haemolytica": "MH", "Mycoplasma bovis": "MB"}
barometer["Pathogen"] = barometer["Pathogen"].replace(short)
barometer["Pathogen"] = barometer["Pathogen_identification"].replace(short).where(barometer["Pathogen_identification"].notna(), barometer["Pathogen"])
result_map = {"Twijfelachtig (PCR)": 1, "POSITIEF": 1, "GEDETECTEERD": 1, "GEDETECTEERD (sterk)": 1, "GEDETECTEERD (zwak)": 1, "GEDETECTEERD (matig)": 1, "GEDETECTEERD (zeer sterk)": 1, "GEDETECTEERD (zeer zwak)": 1, "negatief": 0, "Niet gedetecteerd": 0, "NI": np.nan, "niet interpreteerbaar": np.nan, "Inhibitie": np.nan}
barometer["Result"] = barometer["Result"].map(result_map)
barometer.loc[(barometer["Parameter_code"].eq("BAC_AERO")) & (barometer["Pathogen_result"].isna()), "Result"] = 0
barometer.loc[(barometer["Parameter_code"].eq("BAC_AERO")) & (barometer["Pathogen_result"].notna()), "Result"] = 1
barometer.loc[(barometer["Parameter_code"].eq("BAC_MYCOPLASMA")) & (barometer["Mycoplasma_result"].isna()), "Result"] = np.nan
barometer.loc[(barometer["Parameter_code"].eq("BAC_MYCOPLASMA")) & (barometer["Mycoplasma_result"].eq("neg")), "Result"] = 0
barometer.loc[(barometer["Parameter_code"].eq("BAC_MYCOPLASMA")) & (barometer["Mycoplasma_result"].astype(str).str.contains("POS", na=False)), "Result"] = 1

group_cols = ["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Pathogen", "Date", "Date_month", "Date_week"]
barometer = barometer.groupby(group_cols, dropna=False)["Result"].agg(max_with_na).reset_index()
barometer = barometer[["Lab_reference", "Country", "Breed", "Province", "Farm_ID", "Diagnostic_test", "Sample_type", "Samplenumber", "Date", "Date_month", "Date_week", "Pathogen", "Result"]]
write_delta(barometer, "barometer_dgz")
